In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [2]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [2]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

Loading environment llm_20_questions failed: No module named 'kaggle_environments.envs.llm_20_questions.llm_20_questions'
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 19.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofs

In [3]:
# Run a quick game to see what the observation looks like
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

Player: 0
Angular velocity: 0.0453 rad/turn

Planets (20):
  id=0 owner=Neutral    pos=(96.9, 64.3) r=2.6 ships=68 prod=5
  id=1 owner=Neutral    pos=(35.7, 96.9) r=2.6 ships=68 prod=5
  id=2 owner=Neutral    pos=(64.3, 3.1) r=2.6 ships=68 prod=5
  id=3 owner=Neutral    pos=(3.1, 35.7) r=2.6 ships=68 prod=5
  id=4 owner=Neutral    pos=(94.9, 88.7) r=2.6 ships=62 prod=5
  id=5 owner=Neutral    pos=(11.3, 94.9) r=2.6 ships=62 prod=5


In [ ]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

sun_config = (50.0, 50.0, 10.0)
target_dict = {}

class Target:
    def __init__(self, move, distance, travel_time):
        self.move = move
        self.distance = distance
        self.travel_time = travel_time

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 10, 20)

        travel_time = min_dist / 6
        target = target_dict.get(nearest.id)
        if target != None and target.distance <= 0:
            target_dict.pop(nearest.id)
        elif target != None:
            target.distance = target.distance - 6
            target_dict[nearest.id] = target
            
            
        # Only send if we have enough and there are not ships in transit
        if mine.ships >= ships_needed and target == None:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            
            distance_r = abs((sun_config[0] - mine.x) * math.sin(angle) - (sun_config[1] - mine.y) * math.cos(angle))
            dot_product = ((sun_config[0] - mine.x) * math.cos(angle)) + ((sun_config[1] - mine.y) * math.cos(angle))
            
            move = [mine.id, angle, ships_needed]
            target_dict[nearest.id] = Target(move=move, distance=min_dist, travel_time=travel_time)
            if not (dot_product > 0 and distance_r <= sun_config[2]):
                moves.append(move)

    return moves

In [21]:
# Test it against the random agent
env = make("orbit_wars", debug=True)
env.run([nearest_planet_sniper, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# env.render(mode="ipython", width=800, height=600)
with open("replay.html", "w", encoding="utf-8") as f:
    f.write(env.render(mode="html"))

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE


In [7]:
env4 = make("orbit_wars", debug=True)
env4.run([nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper])

final = env4.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# env4.render(mode="ipython", width=800, height=600)
with open("replay-multi.html", "w", encoding="utf-8") as f:
    f.write(env.render(mode="html"))

Player 0: reward=1, status=DONE
Player 1: reward=1, status=DONE
Player 2: reward=1, status=DONE
Player 3: reward=1, status=DONE


In [ ]:
%%writefile main.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves